# 🏦 Finsight AI: Transaction Category Classification Model & Algorithm Comparison
Welcome to the beginner-friendly Machine Learning notebook for **Finsight AI**!
In this notebook, we build an end-to-end NLP classifier that predicts financial transaction categories from text descriptions, and run a rigorous algorithm comparison across baseline classification algorithms.

### 📌 Pipeline Overview:
1. **Dataset Creation**: Create a balanced prototype dataset across 7 categories.
2. **Data Preprocessing**: Vectorize text using **TF-IDF (Term Frequency-Inverse Document Frequency)**.
3. **Model Training**: Train a **Logistic Regression** baseline classifier.
4. **Evaluation**: Evaluate model accuracy, precision, recall, F1-score, and plot a confusion matrix.
5. **Prediction & Model Export**: Predict categories for unseen transactions and export model artifacts using `joblib`.
6. **Model Comparison Experiment**: Compare Logistic Regression against **Multinomial Naive Bayes** and **Linear Support Vector Machine (Linear SVM)** under identical conditions.

## 1. Import Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
import joblib

# Set plot style
sns.set_theme(style="whitegrid")
np.random.seed(42)

## 2. Dataset Creation & Exploration
We create a representative prototype dataset containing 175 labeled transaction descriptions across 7 categories: `Food`, `Transport`, `Shopping`, `Bills`, `Entertainment`, `Healthcare`, and `Education`.

In [ ]:
data_samples = {
    "Food": [
        "Starbucks coffee and bagel", "McDonalds burger meal", "Dominos pizza delivery",
        "Whole Foods grocery shopping", "Subway lunch combo", "Chipotle burrito bowl",
        "Dunkin Donuts morning coffee", "Taco Bell dinner", "Trader Joes organic vegetables",
        "Local supermarket snacks", "KFC fried chicken box", "Burger King meal deal",
        "Panera Bread soup and salad", "Pizza Hut pepperoni pizza", "Safeway food market",
        "Sushi restaurant dinner", "Bakery fresh bread", "Food truck lunch", "Ice cream parlor",
        "Thai restaurant takeaway", "Italian bistro dinner", "Cafe espresso and muffin",
        "Supermarket fruits and vegetables", "Ramen shop lunch", "Diner breakfast combo"
    ],
    "Transport": [
        "Uber ride to downtown office", "Lyft fare home from airport", "Shell gas station fuel filling",
        "Metro train card recharge", "City bus fare ticket", "Chevron petrol station fill up",
        "Parking garage fee downtown", "Highway toll plaza payment", "Car service station maintenance",
        "Taxi ride to station", "BP gas station station fill", "Electric vehicle charging station",
        "Train ticket intercity express", "Subway fare monthly pass", "Airport shuttle service",
        "Parking meter payment", "Car wash and valet service", "Exxon Mobil gas station",
        "Bike rental hourly pass", "Ferry boat passenger ticket", "Auto repair oil change",
        "Uber Eats driver tip", "Public transport bus pass", "Car insurance premium payment",
        "Speedway gas station refill"
    ],
    "Shopping": [
        "Amazon online purchase electronics", "Target department store clothes", "Walmart household items",
        "Nike store running shoes", "Zara apparel and jacket", "H&M fashion clothes",
        "Best Buy laptop accessory", "Apple Store iPad case", "IKEA furniture assembly items",
        "Sephora cosmetics and skincare", "Nordstrom designer shoes", "eBay online marketplace order",
        "Uniqlo casual wear t-shirts", "Foot Locker athletic shoes", "Home Depot hardware tools",
        "Lululemon workout apparel", "Barnes Noble books purchase", "Etsy handmade craft gift",
        "Gap store denim jeans", "Macy's department store sale", "Adidas sports shoes",
        "TJ Maxx discount clothing", "Micro Center computer parts", "Decathlon sports gear",
        "Urban Outfitters graphic tee"
    ],
    "Bills": [
        "Electricity bill monthly payment", "City water utility bill", "Natural gas heating bill",
        "Comcast internet bill subscription", "Verizon mobile phone bill", "AT&T wireless service bill",
        "Trash collection municipal service", "Apartment monthly rent payment", "Home insurance policy payment",
        "Property tax installment payment", "Sewer utility bill payment", "Solar panel monthly subscription",
        "Spectrum cable TV internet bill", "T-Mobile cell phone bill", "HOA monthly maintenance fee",
        "Mortgage interest installment", "Security system monthly subscription", "Cloud storage monthly plan",
        "Landline telephone service bill", "Health insurance monthly premium", "Dental insurance bill",
        "District heating utility bill", "Utilities electric and water combo", "Mobile data recharge plan",
        "Home broadband internet bill"
    ],
    "Entertainment": [
        "Netflix monthly movie subscription", "Spotify premium music streaming", "AMC Cinema movie tickets",
        "PlayStation Network game download", "Steam store PC game purchase", "YouTube Premium subscription",
        "Disney Plus streaming service", "Concert ticket venue entry", "Bowling alley game night",
        "Hulu streaming subscription", "Audible audiobook subscription", "Nintendo eShop game sale",
        "Xbox Game Pass monthly sub", "Theme park admission ticket", "Music festival ticket pass",
        "Live theater show ticket", "Standup comedy club entry", "Museum entrance ticket fee",
        "Escape room booking ticket", "Twitch channel subscription", "Apple Music monthly plan",
        "VR arcade game session", "Board game cafe bill", "Miniature golf entry fee",
        "IMAX movie tickets pop corn"
    ],
    "Healthcare": [
        "CVS Pharmacy prescription medicine", "Walgreens drug store health check", "Doctor consultation clinic fee",
        "Dental checkup and cleaning", "Eye clinic optometrist visit", "Hospital outpatient consultation",
        "Physical therapy rehab clinic", "Laboratory blood test lab fee", "Urgent care clinic visit",
        "Vision center contact lenses", "Dermatology clinic appointment", "Pharmacy vitamin supplements",
        "Chiropractic adjustment fee", "Orthopedic clinic consultation", "Mental health therapy session",
        "Diagnostic radiology X-ray scan", "Pediatric clinic doctor fee", "Prescription eyeglasses store",
        "First aid kit emergency supplies", "Cardiology specialist checkup", "Medical lab test report",
        "Vaccination clinic immunization", "Surgical clinic consultation", "Hearing aid battery replacement",
        "Wellness clinic wellness check"
    ],
    "Education": [
        "University semester tuition fee", "Coursera online course certificate", "Udemy programming course tutorial",
        "Campus bookstore textbooks purchase", "Student loan monthly payment", "School tuition fee installment",
        "LinkedIn Learning annual subscription", "Coding bootcamp tuition installment", "Language school tuition fee",
        "Academic journal subscription", "College exam fee registration", "Tutoring center monthly fee",
        "STEM workshop registration fee", "GRE test exam registration fee", "IELTS language test fee",
        "Online master degree tuition", "Educational software license", "Piano lessons monthly fee",
        "Art class materials and tuition", "Science lab lab manual textbook", "Library late fee fine",
        "Research paper publishing fee", "High school registration fee", "Certificate exam voucher",
        "Training seminar registration fee"
    ]
}

records = []
for category, desc_list in data_samples.items():
    for desc in desc_list:
        records.append({"description": desc, "category": category})

df = pd.DataFrame(records).sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Total Transactions Loaded: {len(df)}")
df.head(10)

### Visualizing Category Distribution

In [ ]:
plt.figure(figsize=(10, 5))
sns.countplot(data=df, x='category', palette='viridis')
plt.title('Transaction Count by Category', fontsize=14, fontweight='bold')
plt.xlabel('Category')
plt.ylabel('Count')
plt.xticks(rotation=30)
plt.show()

## 3. Data Splitting & TF-IDF Vectorization
We split the dataset into 80% Training and 20% Testing sets using stratified sampling. Then we convert text into numerical feature vectors using `TfidfVectorizer`.

In [ ]:
X = df['description']
y = df['category']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Training set size: {len(X_train)}")
print(f"Testing set size:  {len(X_test)}")

# Preprocess & Vectorize
vectorizer = TfidfVectorizer(stop_words='english', lowercase=True, ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(f"TF-IDF Vocabulary Size: {len(vectorizer.get_feature_names_out())}")

## 4. Baseline Model Training
We train our baseline **Logistic Regression** classifier on the TF-IDF feature vectors.

In [ ]:
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train_tfidf, y_train)
print("Logistic Regression baseline model trained successfully!")

## 5. Baseline Evaluation
We evaluate the baseline model on the unseen test set using Accuracy, Precision, Recall, F1-Score, and a Confusion Matrix heatmap.

In [ ]:
y_pred = model.predict(X_test_tfidf)

acc = accuracy_score(y_test, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='weighted')

print("===========================================")
print(f" Accuracy:  {acc * 100:.2f}%")
print(f" Precision: {precision:.4f}")
print(f" Recall:    {recall:.4f}")
print(f" F1-Score:  {f1:.4f}")
print("===========================================\n")
print(classification_report(y_test, y_pred))

### Confusion Matrix Visualization

In [ ]:
labels = sorted(df['category'].unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.title('Confusion Matrix - Transaction Classifier', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Category')
plt.ylabel('Actual Category')
plt.show()

## 6. Saving Model Artifacts & Testing Predictions

In [ ]:
os.makedirs('../models', exist_ok=True)
joblib.dump(model, '../models/transaction_classifier.joblib')
joblib.dump(vectorizer, '../models/tfidf_vectorizer.joblib')
print("Saved model and vectorizer to models/ directory.")

# Example predictions
sample_txs = [
    "Starbucks iced caramel macchiato",
    "Uber ride to airport terminal 2",
    "Amazon electronics order USB-C cable",
    "Electricity utility bill auto debit",
    "Spotify music subscription premium",
    "Doctor consultation fee and blood test",
    "Udemy Python machine learning course"
]

sample_tfidf = vectorizer.transform(sample_txs)
preds = model.predict(sample_tfidf)

print("\n--- Real-Time Prediction Test ---")
for tx, pred in zip(sample_txs, preds):
    print(f"Description: '{tx}' --> Category: {pred}")

---

## 7. Model Comparison Experiment
To determine if alternative classification algorithms can achieve superior text categorization accuracy on sparse TF-IDF vectors, we compare **3 algorithms** using the **exact same 80/20 train/test dataset split**:

### Why Each Algorithm Was Selected:
1. **Logistic Regression (Baseline)**: A linear probability model that estimates class likelihoods. Extremely fast and highly interpretable for text.
2. **Multinomial Naive Bayes**: A probabilistic classifier based on Bayes' theorem assuming word independence. Highly effective for word-frequency and TF-IDF text features.
3. **Linear Support Vector Machine (Linear SVM)**: A maximum-margin classifier that finds the optimal boundary line/hyperplane separating text classes in high-dimensional feature spaces.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000),
    "Multinomial Naive Bayes": MultinomialNB(),
    "Linear SVM": LinearSVC(random_state=42, max_iter=1000)
}

comparison_results = []
for name, clf in models.items():
    clf.fit(X_train_tfidf, y_train)
    preds = clf.predict(X_test_tfidf)
    
    acc_c = accuracy_score(y_test, preds)
    prec_c, rec_c, f1_c, _ = precision_recall_fscore_support(y_test, preds, average='weighted')
    
    comparison_results.append({
        "Model": name,
        "Accuracy": acc_c,
        "Precision": prec_c,
        "Recall": rec_c,
        "F1-Score": f1_c
    })

comp_df = pd.DataFrame(comparison_results)
comp_df

### Visualizing Model Comparison Results

In [ ]:
plt.figure(figsize=(10, 6))
df_melted = pd.melt(comp_df, id_vars=['Model'], var_name='Metric', value_name='Score')
sns.barplot(data=df_melted, x='Metric', y='Score', hue='Model', palette='magma')
plt.title('Algorithm Comparison across Performance Metrics', fontsize=14, fontweight='bold')
plt.ylim(0, 1.05)
plt.legend(loc='lower right')
plt.show()